In [0]:
spark.sql("""
CREATE OR REPLACE TABLE workspace.delhi_air.silver_weather_hourly AS
WITH parsed AS (
    SELECT ingested_at,
           from_json(raw_json, 'hourly struct<time:array<string>, temperature_2m:array<double>, relative_humidity_2m:array<double>, wind_speed_10m:array<double>, wind_direction_10m:array<double>, surface_pressure:array<double>, precipitation:array<double>, boundary_layer_height:array<double>>') AS j
    FROM workspace.delhi_air.bronze_openmeteo
    WHERE source = 'weather'
),
exploded AS (
    SELECT ingested_at,
           explode(arrays_zip(j.hourly.time, j.hourly.temperature_2m, j.hourly.relative_humidity_2m, j.hourly.wind_speed_10m, j.hourly.wind_direction_10m, j.hourly.surface_pressure, j.hourly.precipitation, j.hourly.boundary_layer_height)) AS z
    FROM parsed
),
flat AS (
    SELECT CAST(z.time AS TIMESTAMP) AS hour_ist, ingested_at,
           z.temperature_2m AS temperature_c,
           z.relative_humidity_2m AS humidity_pct,
           z.wind_speed_10m AS wind_speed_kmh,
           z.wind_direction_10m AS wind_direction_deg,
           z.surface_pressure AS pressure_hpa,
           z.precipitation AS precipitation_mm,
           z.boundary_layer_height AS boundary_layer_m
    FROM exploded
),
ranked AS (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY hour_ist ORDER BY ingested_at DESC) AS rn
    FROM flat
    WHERE hour_ist < date_trunc('hour', ingested_at + INTERVAL 330 MINUTES)
)
SELECT hour_ist, temperature_c, humidity_pct, wind_speed_kmh, wind_direction_deg,
       pressure_hpa, precipitation_mm, boundary_layer_m
FROM ranked
WHERE rn = 1
""")

spark.sql("""
CREATE OR REPLACE TABLE workspace.delhi_air.silver_model_pm25_hourly AS
WITH parsed AS (
    SELECT ingested_at,
           from_json(raw_json, 'hourly struct<time:array<string>, pm2_5:array<double>>') AS j
    FROM workspace.delhi_air.bronze_openmeteo
    WHERE source = 'air_quality'
),
exploded AS (
    SELECT ingested_at,
           explode(arrays_zip(j.hourly.time, j.hourly.pm2_5)) AS z
    FROM parsed
),
flat AS (
    SELECT CAST(z.time AS TIMESTAMP) AS hour_ist, ingested_at, z.pm2_5 AS model_pm25_ugm3
    FROM exploded
),
ranked AS (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY hour_ist ORDER BY ingested_at DESC) AS rn
    FROM flat
    WHERE hour_ist < date_trunc('hour', ingested_at + INTERVAL 330 MINUTES)
)
SELECT hour_ist, model_pm25_ugm3
FROM ranked
WHERE rn = 1
""")

for t in ["silver_weather_hourly", "silver_model_pm25_hourly"]:
    spark.sql(f"SELECT '{t}' AS tbl, count(*) AS hours, min(hour_ist) AS first_hour, max(hour_ist) AS last_hour FROM workspace.delhi_air.{t}").show(truncate=False)